# einops-reduce — ex9: top-k via repeated masked max (integrative multi-step)

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `einops-reduce`. Running the final beacon cell reports progress against the `Einops: Reduce` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Einops: Reduce` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`einops-reduce`** (exercise 9). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "einops-reduce"
DD_SUBTOPIC = "Einops: Reduce"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## einops.reduce — quick refresher

`reduce(tensor, pattern, op)` collapses one or more named axes with a reduction `op` ∈ `{'mean', 'sum', 'max', 'min', 'prod'}`. Drop an axis name on the right side to reduce it; keep it inside parentheses on the left and decompose first to do windowed pooling.

The exercises below stop being about *which op?* and start being about *reduce as part of a larger pipeline* — pyramid pooling, per-channel normalization, argmax-without-`torch.argmax`, top-k by repeated masked max. Each one needs visualization or print-debug to be solvable in your head.

### Exercise 9 — top-k via repeated masked max (integrative multi-step)

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Create
> LO: Combine reduce-max, boolean masking, and iteration to implement top-k from scratch, visualizing the mask after each iteration to see what got pulled out.
> Keywords: top-k, iterative, mask, reduce-max
> ```

**KCs targeted:** `reduce-max`, `reduce-keepdim-with-parens`

This integrates ex8's argmax recipe into a real algorithm: **top-k via repeated masked max**. The idea: find the max, record its position, mask it out, repeat `k` times. Slower than `torch.topk` for big `k`, but it's a transparent algorithm and the exact shape of code you'd write to implement custom routing (MoE top-k gating, beam search, k-WTA layers).

Implement `ex9_topk_via_reduce(x, k)` for a 1-D tensor `x` of shape `(N,)`, returning `(values, indices)` both shape `(k,)`, in **descending order of value**, **without calling `torch.topk`** or `.topk`.

Recipe:
1. Start with a `mask` of all `True`, shape `(N,)`.
2. For each iteration `i in 0..k-1`:
   a. Use `reduce(masked_x, 'n -> ()', 'max')` (where `masked_x` has `-inf` at masked positions) to get the current max value.
   b. Find its index by comparing `x == max_val` and taking the first match within the still-`True` mask (use the same `-1` trick from ex8).
   c. Record value+index, then set `mask[chosen_idx] = False`.
3. Plot `mask.float()` after each iteration as a row in a heatmap (shape `(k, N)`) — that's the visualization that makes the algorithm legible.

Return `(values, indices)` as a tuple of 1-D tensors of length `k`.

In [ ]:
def ex9_topk_via_reduce(x: Tensor, k: int) -> tuple[Tensor, Tensor]:
    import matplotlib.pyplot as plt
    N = x.shape[0]
    mask = t.ones(N, dtype=t.bool, device=x.device)
    values = []
    indices = []
    mask_history = []
    neg_inf = t.tensor(float('-inf'), device=x.device, dtype=x.dtype)
    for _ in range(k):
        masked_x = t.where(mask, x, neg_inf)
        # reduce-max over the single axis; '() -> ()' keeps no axis.
        max_val = reduce(masked_x, 'n -> ()', 'max').squeeze()
        # Find first index where x == max_val AND still unmasked.
        is_max_here = (masked_x == max_val)
        col_idx = t.arange(N, device=x.device)
        # Pick the *leftmost* surviving max via min over true-positions.
        masked_idx = t.where(is_max_here, col_idx, t.full_like(col_idx, N))
        chosen = reduce(masked_idx, 'n -> ()', 'min').squeeze().long()
        values.append(max_val)
        indices.append(chosen)
        mask[chosen] = False
        mask_history.append(mask.clone().float())
    history = t.stack(mask_history)  # (k, N)
    fig, ax = plt.subplots(figsize=(8, max(2, k * 0.4)))
    ax.imshow(history.cpu().numpy(), aspect='auto', cmap='gray')
    ax.set_xlabel('position')
    ax.set_ylabel('iteration')
    ax.set_title(f'mask state after each of {k} iterations (white = still eligible)')
    plt.tight_layout()
    plt.show()
    return t.stack(values), t.stack(indices).long()


<details><summary>Solution</summary>

```python
def ex9_topk_via_reduce(x: Tensor, k: int) -> tuple[Tensor, Tensor]:
    import matplotlib.pyplot as plt
    N = x.shape[0]
    mask = t.ones(N, dtype=t.bool, device=x.device)
    values = []
    indices = []
    mask_history = []
    neg_inf = t.tensor(float('-inf'), device=x.device, dtype=x.dtype)
    for _ in range(k):
        masked_x = t.where(mask, x, neg_inf)
        # reduce-max over the single axis; '() -> ()' keeps no axis.
        max_val = reduce(masked_x, 'n -> ()', 'max').squeeze()
        # Find first index where x == max_val AND still unmasked.
        is_max_here = (masked_x == max_val)
        col_idx = t.arange(N, device=x.device)
        # Pick the *leftmost* surviving max via min over true-positions.
        masked_idx = t.where(is_max_here, col_idx, t.full_like(col_idx, N))
        chosen = reduce(masked_idx, 'n -> ()', 'min').squeeze().long()
        values.append(max_val)
        indices.append(chosen)
        mask[chosen] = False
        mask_history.append(mask.clone().float())
    history = t.stack(mask_history)  # (k, N)
    fig, ax = plt.subplots(figsize=(8, max(2, k * 0.4)))
    ax.imshow(history.cpu().numpy(), aspect='auto', cmap='gray')
    ax.set_xlabel('position')
    ax.set_ylabel('iteration')
    ax.set_title(f'mask state after each of {k} iterations (white = still eligible)')
    plt.tight_layout()
    plt.show()
    return t.stack(values), t.stack(indices).long()
```

**Why this matches `torch.topk` exactly.** Both algorithms break ties by leftmost index (we use `min` over surviving-true index positions, which is the leftmost). And both return values in descending order because we pull the max out each iteration.

**Why visualize the mask.** Each row of the heatmap is a snapshot of "who's still eligible". You see a single pixel flip black each iteration. This is exactly the picture in your head when you debug a routing layer that's eliminating the wrong tokens.

**Performance note.** This is O(k·N). Real `topk` uses a partial sort and is O(N log k). For `k ≪ N`, the difference doesn't matter; for `k ≈ N`, prefer `sort`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex9'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex9',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()